# VHAGAR — Prithvi fine-tune on Colab

Run this notebook top to bottom on a **GPU runtime** (Runtime → Change runtime type → T4 GPU).
It fine-tunes Prithvi-EO-2.0-300M on the six-band burn-scar chips you exported locally and
writes per-chip predictions. Building the chips and the final leakage-safe scoring
(`vhagar t2-headtohead`) run on your own machine, not here. See `docs/16_COLAB_PRITHVI.md`.

**Before you start:** locally run `t2-prithvi-build` then `t2-prithvi-export` (or reuse an
existing `data/t2_prithvi_chips/`). Bundle the chips + config into ONE zip on your machine
(reliable; uploading 1000+ loose files to Drive is not), and put it in Drive at
`MyDrive/vhagar/prithvi_colab_bundle.zip`:

```powershell
Compress-Archive -Path .\data\t2_prithvi_chips, .\prithvi_burnscars_vhagar.yaml `
    -DestinationPath .\prithvi_colab_bundle.zip
```


## 1. Confirm the GPU


In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
!mkdir -p /content/work
!cp /content/drive/MyDrive/vhagar/prithvi_colab_bundle.zip /content/work/
!cd /content/work && unzip -q -o prithvi_colab_bundle.zip
os.chdir('/content/work')
# The zip holds data/t2_prithvi_chips/... + the yaml at the paths the config expects.
!ls data/t2_prithvi_chips && echo '---' && ls data/t2_prithvi_chips/splits


## 3. Install TerraTorch

The pretrained backbone downloads from Hugging Face on first use. If it is rate-limited,
uncomment the `login()` cell and paste a read token from huggingface.co/settings/tokens.


In [ ]:
!pip install -q terratorch


In [ ]:
# from huggingface_hub import login
# login()


## 4. Fine-tune

Early-stops on `val/loss` (patience 15, max 100 epochs). Roughly 20–40 min on a T4 for a
~470-chip training set. If it errors on a datamodule argument, that is TerraTorch version
drift — check `!terratorch fit --help` for the current arg names.


In [ ]:
!terratorch fit -c prithvi_burnscars_vhagar.yaml


In [ ]:
import glob
ckpt = sorted(glob.glob('lightning_logs/version_*/checkpoints/*.ckpt'))[-1]
print('checkpoint:', ckpt)


## 5. Predict the held-out test chips

One mask per chip; the local scorer stitches them back per fire via `_chips.json`.


In [ ]:
!terratorch predict -c prithvi_burnscars_vhagar.yaml --ckpt_path {ckpt} \
    --predict_output_dir /content/work/preds \
    --data.init_args.predict_data_root /content/work/data/t2_prithvi_chips/data \
    --data.init_args.predict_split     /content/work/data/t2_prithvi_chips/splits/test.txt
!ls /content/work/preds | head


### Normalize prediction filenames

The scorer maps a prediction to a chip by reducing its stem to the chip stem, tolerating one
trailing suffix. If TerraTorch stacked two (e.g. `{chipstem}_merged_pred.tif`), collapse them
to `{chipstem}.tif`. This cell is a no-op if names are already clean.


In [ ]:
import os, re
d = '/content/work/preds'
for f in os.listdir(d):
    new = re.sub(r'_merged(_pred)?\.tif$', '.tif', f)
    if new != f:
        os.rename(os.path.join(d, f), os.path.join(d, new))
print(sorted(os.listdir(d))[:5])


## 6. Copy predictions back to Drive

Then sync/download `MyDrive/vhagar/prithvi_preds` to your machine as `data/prithvi_preds/`.


In [ ]:
!cp -r /content/work/preds /content/drive/MyDrive/vhagar/prithvi_preds
print('done — predictions are in MyDrive/vhagar/prithvi_preds')


## 7. Score locally (not in Colab)

Back in the repo on your machine:

```
vhagar t2-headtohead --cache-dir data/t2_prithvi --pred-dir data/prithvi_preds \
    --chips-manifest data/t2_prithvi_chips/_chips.json \
    --split data/t2_prithvi_chips/_split.json --out-json h2h_report.json
```

Prithvi vs U-Net vs RBR on the identical held-out fires, with paired-bootstrap CIs. The split
is authoritative and in-sample Prithvi masks are rejected, so the margins cannot be leakage
artefacts.
